# Airline Disruption Forecasting Agent Demo

This notebook demonstrates a high-level demo of the airline disruption forecasting and decision-support agent using multiple synthetic disruption scenarios from a CSV file.


## 1. Load Scenario Data
The CSV contains several synthetic scenarios with weather, crew, aircraft, and network signals.


In [ ]:
import pandas as pd

scenarios = pd.read_csv('../examples/disruption_scenarios.csv')
scenarios


## 2. Define Risk Scoring and Driver Logic
This simplified scoring logic represents the core reasoning behavior from the prototype.


In [ ]:
def evaluate_risk(row):
    score = 0
    if 'storm' in row['weather']:
        score += 0.3
    if 'severe' in row['weather']:
        score += 0.2
    if 'limited' in row['crew']:
        score += 0.4
    if 'delay' in row['aircraft'] or 'maintenance' in row['aircraft']:
        score += 0.1
    if 'delay' in row['network']:
        score += 0.2
    return min(round(score, 2), 1.0)

def identify_driver(row):
    if 'limited' in row['crew']:
        return 'crew constraints'
    elif 'storm' in row['weather']:
        return 'weather conditions'
    elif 'delay' in row['network']:
        return 'network propagation'
    elif 'delay' in row['aircraft'] or 'maintenance' in row['aircraft']:
        return 'aircraft constraints'
    return 'normal operations'

def recommend_action(driver):
    actions = {
        'crew constraints': 'Reassign reserve crew to high-priority flights',
        'weather conditions': 'Delay departures and adjust routing',
        'network propagation': 'Protect key inbound connections',
        'aircraft constraints': 'Swap aircraft if available',
        'normal operations': 'Continue monitoring'
    }
    return actions.get(driver, 'Continue monitoring')


## 3. Run the Agent Logic Across All Scenarios
This simulates the agent applying the same decision logic across multiple operational cases.


In [ ]:
results = []

for _, row in scenarios.iterrows():
    risk_score = evaluate_risk(row)
    driver = identify_driver(row)
    recommendation = recommend_action(driver)
    confidence = 0.72 if risk_score >= 0.7 else 0.65
    results.append({
        'scenario_id': row['scenario_id'],
        'airport': row['airport'],
        'risk_score': risk_score,
        'dominant_driver': driver,
        'recommendation': recommendation,
        'confidence': confidence,
        'matches_expected_driver': driver == row['expected_dominant_driver']
    })

results_df = pd.DataFrame(results)
results_df


## 4. Summarize Demo Results
The demo shows whether the simplified agent identifies the expected dominant driver across multiple synthetic cases.


In [ ]:
accuracy = results_df['matches_expected_driver'].mean()
print(f'Dominant driver match rate: {accuracy:.0%}')


## 5. How to Explain This in the Presentation
In the recorded demo, briefly show that the notebook loads multiple scenarios from a CSV, applies risk scoring and driver identification, then produces a structured output table with risk score, dominant driver, recommendation, and confidence. This supports the system claims with multiple examples rather than a single hand-picked case.
